# 🔬 OncosenseAI — Module 3: Treatment Matcher

[![Module](https://img.shields.io/badge/Module-3%20Treatment%20Matcher-purple?style=for-the-badge)](#)
[![Guidelines](https://img.shields.io/badge/Guidelines-NCCN%20%7C%20NICE%20%7C%20WHO-blue?style=for-the-badge)](#)
[![Colab](https://img.shields.io/badge/Run%20In-Google%20Colab-orange?style=for-the-badge)](#)

> **What this module does:**  
> Takes cancer type + stage + ECOG + genomic markers → outputs guideline-mapped treatment protocol + clinical trial eligibility

```
Module 1 risk score + Module 2 visual findings
                    │
                    ▼
       Cancer type + Stage (I–IV)
                    │
       ┌────────────┼────────────┐
       ▼            ▼            ▼
  Treatment    ECOG logic   Genomic
  Protocol     (fitness)    Markers
       │            │            │
       └────────────┼────────────┘
                    ▼
     Final Treatment Recommendation
     + Clinical Trial Eligibility
     + module3_output.json
```

> ⚠️ Research Prototype Only. Not validated for clinical use. Not FDA/CE approved.  
> Always apply clinical judgement. These are guideline references, not prescriptions.

---
### How to run
Run cells one by one top to bottom. No API key needed for core treatment matching.  
ClinicalTrials.gov uses their free public API — no key required.

In [1]:
# Cell 1 — Install dependencies
!pip install requests pandas -q

import json
import requests
import pandas as pd
from google.colab import files
from IPython.display import display

print('✅ Dependencies loaded')
print('   No API key needed — ClinicalTrials.gov is a free public API')

✅ Dependencies loaded
   No API key needed — ClinicalTrials.gov is a free public API


In [2]:
# Cell 2 — NCCN/NICE/WHO Treatment Guidelines Database
# Source: NCCN Guidelines, NICE NG12/NG151, ESMO Clinical Practice Guidelines
# Last reviewed: 2024

TREATMENT_DB = {

    # ── COLORECTAL CANCER ──────────────────────────────────────────────────
    ('colorectal', 'I'): {
        'primary': 'Surgical resection (colectomy with lymph node dissection)',
        'adjuvant': 'No adjuvant chemotherapy for Stage I (low risk)',
        'regimens': ['Colectomy (laparoscopic or open)'],
        'guideline': 'NCCN Colon Cancer v2.2024 / NICE NG151',
        'notes': 'Surveillance colonoscopy at 1 year post-resection.'
    },
    ('colorectal', 'II'): {
        'primary': 'Surgical resection ± adjuvant chemotherapy (high-risk features)',
        'adjuvant': 'CAPOX or FOLFOX x6 months if high-risk (T4, perforation, <12 nodes)',
        'regimens': ['CAPOX (capecitabine + oxaliplatin)', 'FOLFOX (5-FU + leucovorin + oxaliplatin)'],
        'guideline': 'NCCN Colon Cancer v2.2024',
        'notes': 'MSI-H Stage II — excellent prognosis, adjuvant chemotherapy may not benefit.'
    },
    ('colorectal', 'III'): {
        'primary': 'Surgical resection + adjuvant chemotherapy',
        'adjuvant': 'CAPOX x6 months OR FOLFOX x6 months (standard of care)',
        'regimens': ['CAPOX', 'FOLFOX', 'FOLFIRI (if oxaliplatin intolerant)'],
        'guideline': 'NCCN Colon Cancer v2.2024 / ESMO 2023',
        'notes': 'MSI-H/dMMR — consider immunotherapy (pembrolizumab) in eligible patients.'
    },
    ('colorectal', 'IV'): {
        'primary': 'Systemic chemotherapy ± targeted therapy ± surgery (if resectable mets)',
        'adjuvant': 'FOLFOX + bevacizumab OR FOLFIRI + bevacizumab (first-line)',
        'regimens': ['FOLFOX + bevacizumab', 'FOLFIRI + bevacizumab', 'FOLFOXIRI + bevacizumab (fit patients)', 'Pembrolizumab (MSI-H/dMMR first-line)', 'Cetuximab/panitumumab (RAS/RAF wild-type)'],
        'guideline': 'NCCN Colon Cancer v2.2024 / ESMO 2023',
        'notes': 'RAS/RAF/MSI status mandatory before treatment. Liver-limited disease: consider resection after downstaging.'
    },

    # ── GASTRIC / STOMACH CANCER ───────────────────────────────────────────
    ('gastric', 'I'): {
        'primary': 'Surgical resection (D2 gastrectomy)',
        'adjuvant': 'Adjuvant chemotherapy for Stage IB+ (S-1 monotherapy in Asian populations)',
        'regimens': ['D2 gastrectomy', 'S-1 (tegafur/gimeracil/oteracil) — adjuvant'],
        'guideline': 'NCCN Gastric v2.2024 / ESMO 2022',
        'notes': 'ESD (endoscopic submucosal dissection) for T1a mucosal disease.'
    },
    ('gastric', 'II'): {
        'primary': 'Perioperative chemotherapy + D2 gastrectomy (preferred) OR surgery + adjuvant',
        'adjuvant': 'FLOT4 perioperative (preferred in fit patients) OR CAPOX adjuvant',
        'regimens': ['FLOT4 (5-FU + leucovorin + oxaliplatin + docetaxel)', 'CAPOX adjuvant', 'ECF/ECX (older regimen)'],
        'guideline': 'NCCN Gastric v2.2024 / FLOT4 trial',
        'notes': 'FLOT4 now preferred over ECF perioperatively (FLOT4 trial, Al-Batran 2019).'
    },
    ('gastric', 'III'): {
        'primary': 'Perioperative FLOT4 + D2 gastrectomy',
        'adjuvant': 'FLOT4 x4 cycles pre-op + surgery + FLOT4 x4 cycles post-op',
        'regimens': ['FLOT4 (standard)', 'CAPOX (alternative)', 'Nivolumab + chemotherapy (HER2-negative, CPS≥5)'],
        'guideline': 'NCCN Gastric v2.2024 / ESMO 2022 / CheckMate-649',
        'notes': 'HER2 testing mandatory. PD-L1 CPS score guides immunotherapy eligibility.'
    },
    ('gastric', 'IV'): {
        'primary': 'Systemic chemotherapy + targeted/immunotherapy',
        'adjuvant': 'Nivolumab + FOLFOX/CAPOX (HER2-negative) OR trastuzumab + CAPOX/FP (HER2+)',
        'regimens': ['Nivolumab + FOLFOX (CheckMate-649)', 'Trastuzumab + CAPOX (HER2+ — ToGA trial)', 'Pembrolizumab + chemo (KEYNOTE-590)', 'Ramucirumab + paclitaxel (2nd line)'],
        'guideline': 'NCCN Gastric v2.2024 / ESMO 2022',
        'notes': 'HER2, PD-L1 CPS, MSI, EBV status mandatory. Best supportive care for ECOG 3-4.'
    },

    # ── PANCREATIC CANCER ─────────────────────────────────────────────────
    ('pancreatic', 'I'): {
        'primary': 'Surgical resection (Whipple / distal pancreatectomy) + adjuvant chemotherapy',
        'adjuvant': 'Modified FOLFIRINOX x6 months (preferred, fit patients) OR gemcitabine + capecitabine',
        'regimens': ['Modified FOLFIRINOX (mFOLFIRINOX)', 'Gemcitabine + capecitabine', 'Gemcitabine monotherapy (ECOG 2+)'],
        'guideline': 'NCCN Pancreatic v2.2024 / ESPAC-4 trial',
        'notes': 'Only ~20% are resectable at diagnosis. Neoadjuvant therapy increasingly used for borderline resectable.'
    },
    ('pancreatic', 'II'): {
        'primary': 'Borderline resectable: neoadjuvant chemotherapy → reassess resectability',
        'adjuvant': 'FOLFIRINOX neoadjuvant → surgery → adjuvant gemcitabine OR gemcitabine + nab-paclitaxel',
        'regimens': ['FOLFIRINOX neoadjuvant', 'Gemcitabine + nab-paclitaxel', 'Gemcitabine + capecitabine adjuvant'],
        'guideline': 'NCCN Pancreatic v2.2024',
        'notes': 'Multidisciplinary team review essential for borderline resectable cases.'
    },
    ('pancreatic', 'III'): {
        'primary': 'Locally advanced unresectable: systemic chemotherapy ± chemoradiation',
        'adjuvant': 'FOLFIRINOX OR gemcitabine + nab-paclitaxel → reassess for resectability',
        'regimens': ['FOLFIRINOX', 'Gemcitabine + nab-paclitaxel (Abraxane)', 'Chemoradiation (selected cases after chemotherapy)'],
        'guideline': 'NCCN Pancreatic v2.2024 / LAP07 trial',
        'notes': 'BRCA1/2 germline testing for all patients — PARP inhibitor maintenance if BRCA+.'
    },
    ('pancreatic', 'IV'): {
        'primary': 'Palliative systemic chemotherapy',
        'adjuvant': 'FOLFIRINOX (fit, ECOG 0-1) OR gemcitabine + nab-paclitaxel (ECOG 0-2)',
        'regimens': ['FOLFIRINOX (fit patients)', 'Gemcitabine + nab-paclitaxel', 'Gemcitabine monotherapy (ECOG 2+)', 'Olaparib maintenance (BRCA1/2+, after platinum)', 'Pembrolizumab (MSI-H/dMMR)'],
        'guideline': 'NCCN Pancreatic v2.2024 / POLO trial / MPACT trial',
        'notes': 'Median OS ~11 months with FOLFIRINOX. Biliary stenting for obstruction. Early palliative care.'
    },

    # ── ESOPHAGEAL CANCER ─────────────────────────────────────────────────
    ('esophageal', 'I'): {
        'primary': 'Endoscopic resection (T1a) OR surgical resection (T1b)',
        'adjuvant': 'No adjuvant for Stage I after complete resection',
        'regimens': ['ESD/EMR (mucosal disease)', 'Esophagectomy (submucosal or deeper)'],
        'guideline': 'NCCN Esophageal v2.2024 / NICE NG83',
        'notes': 'Barrett esophagus surveillance for adenocarcinoma. SCC: location determines approach.'
    },
    ('esophageal', 'II'): {
        'primary': 'Neoadjuvant chemoradiation (CROSS regimen) + esophagectomy',
        'adjuvant': 'CROSS: carboplatin + paclitaxel + RT → surgery → nivolumab adjuvant (if residual disease)',
        'regimens': ['CROSS (carboplatin + paclitaxel + 41.4 Gy RT)', 'Nivolumab adjuvant (CheckMate-577, residual disease)'],
        'guideline': 'NCCN Esophageal v2.2024 / CROSS trial / CheckMate-577',
        'notes': 'CROSS trial established as standard of care for resectable esophageal/GEJ cancer.'
    },
    ('esophageal', 'III'): {
        'primary': 'Neoadjuvant chemoradiation + surgery OR definitive chemoradiation',
        'adjuvant': 'CROSS neoadjuvant → esophagectomy → nivolumab adjuvant if ypN+ or R1',
        'regimens': ['CROSS regimen', 'Nivolumab adjuvant (CheckMate-577)', 'FOLFOX (adenocarcinoma alternative)', 'Definitive CRT (SCC — if surgical risk high)'],
        'guideline': 'NCCN Esophageal v2.2024',
        'notes': 'Surgical fitness assessment critical. High-volume centre referral recommended.'
    },
    ('esophageal', 'IV'): {
        'primary': 'Systemic chemotherapy + immunotherapy',
        'adjuvant': 'Nivolumab + FOLFOX/CAPOX (adenocarcinoma) OR nivolumab + ipilimumab (SCC, CPS≥1)',
        'regimens': ['Nivolumab + chemotherapy (CheckMate-649)', 'Nivolumab + ipilimumab (SCC)', 'Pembrolizumab + chemo (KEYNOTE-590)', 'Trastuzumab + chemo (HER2+ GEJ adenocarcinoma)'],
        'guideline': 'NCCN Esophageal v2.2024 / KEYNOTE-590',
        'notes': 'HER2, PD-L1 CPS, MSI testing mandatory. Stenting/dilation for dysphagia palliation.'
    },
}

CANCER_TYPES = ['colorectal', 'gastric', 'pancreatic', 'esophageal']
STAGES = ['I', 'II', 'III', 'IV']

print(f'✅ Treatment database loaded')
print(f'   Cancer types: {CANCER_TYPES}')
print(f'   Stages: {STAGES}')
print(f'   Total protocols: {len(TREATMENT_DB)}')
print(f'   Guidelines: NCCN 2024 · NICE · ESMO 2023 · WHO')

✅ Treatment database loaded
   Cancer types: ['colorectal', 'gastric', 'pancreatic', 'esophageal']
   Stages: ['I', 'II', 'III', 'IV']
   Total protocols: 16
   Guidelines: NCCN 2024 · NICE · ESMO 2023 · WHO


In [3]:
# Cell 3 — ECOG Performance Status De-escalation Logic

ECOG_DESCRIPTIONS = {
    0: 'Fully active — no restrictions',
    1: 'Restricted in strenuous activity — ambulatory, light work',
    2: 'Ambulatory >50% of waking hours — self-care, no work',
    3: 'Limited self-care — confined to bed/chair >50% of waking hours',
    4: 'Completely disabled — no self-care, confined to bed/chair'
}

def apply_ecog_deescalation(protocol, ecog_score):
    modified = dict(protocol)
    notes_extra = ''

    if ecog_score == 0:
        notes_extra = 'ECOG 0 — full treatment intensity appropriate.'

    elif ecog_score == 1:
        notes_extra = 'ECOG 1 — standard treatment. Monitor closely for toxicity.'

    elif ecog_score == 2:
        notes_extra = (
            'ECOG 2 — de-escalate treatment intensity. '
            'Avoid FOLFIRINOX/FLOT4. Prefer gemcitabine monotherapy or capecitabine. '
            'Dose reduce by 20-25%. Early palliative care referral.'
        )
        # De-escalate regimens
        safe_regimens = [r for r in modified.get('regimens', [])
                         if not any(x in r for x in ['FOLFIRINOX', 'FLOT4', 'FOLFOXIRI'])]
        if safe_regimens:
            modified['regimens'] = safe_regimens
        modified['adjuvant'] = 'Dose-reduced or single-agent chemotherapy — full combination may be too toxic'

    elif ecog_score == 3:
        notes_extra = (
            'ECOG 3 — chemotherapy generally NOT recommended. '
            'Best supportive care (BSC) + symptom management. '
            'Single-agent capecitabine or gemcitabine only if strong rationale. '
            'Urgent palliative care referral.'
        )
        modified['primary'] = 'Best supportive care (BSC)'
        modified['adjuvant'] = 'Palliative/symptomatic treatment only'
        modified['regimens'] = ['Best supportive care', 'Palliative radiotherapy (pain/bleeding)', 'Symptom management']

    elif ecog_score == 4:
        notes_extra = (
            'ECOG 4 — chemotherapy contraindicated. '
            'Best supportive care only. Hospice/palliative care referral.'
        )
        modified['primary'] = 'Hospice / palliative care'
        modified['adjuvant'] = 'Comfort measures only'
        modified['regimens'] = ['Hospice care', 'Pain management', 'Symptom control']

    modified['ecog_note'] = notes_extra
    return modified

print('✅ ECOG de-escalation logic loaded')
for k, v in ECOG_DESCRIPTIONS.items():
    print(f'   ECOG {k}: {v}')

✅ ECOG de-escalation logic loaded
   ECOG 0: Fully active — no restrictions
   ECOG 1: Restricted in strenuous activity — ambulatory, light work
   ECOG 2: Ambulatory >50% of waking hours — self-care, no work
   ECOG 3: Limited self-care — confined to bed/chair >50% of waking hours
   ECOG 4: Completely disabled — no self-care, confined to bed/chair


In [4]:
# Cell 4 — Genomic Marker Integration

def apply_genomic_modifiers(protocol, cancer_type, stage, markers):
    modified = dict(protocol)
    genomic_notes = []
    extra_regimens = []

    msi_h  = markers.get('MSI_H', False)
    her2   = markers.get('HER2', False)
    kras   = markers.get('KRAS', False)   # KRAS mutant
    braf   = markers.get('BRAF', False)   # BRAF V600E
    brca   = markers.get('BRCA', False)   # BRCA1/2 germline
    pdl1   = markers.get('PDL1_CPS', 0)  # PD-L1 CPS score

    # MSI-H / dMMR — immunotherapy eligible across all cancer types
    if msi_h:
        genomic_notes.append(
            '🧬 MSI-H/dMMR detected — pembrolizumab eligible (FDA approved all solid tumours, stage IV). '
            'Stage II colorectal: excellent prognosis, adjuvant chemo may not benefit. '
            'Stage III-IV: pembrolizumab first-line (KEYNOTE-158/177).'
        )
        extra_regimens.append('Pembrolizumab (MSI-H/dMMR — KEYNOTE-158)')
        if stage == 'IV':
            extra_regimens.append('Dostarlimab (MSI-H/dMMR alternative)')

    # HER2 — gastric and esophageal
    if her2 and cancer_type in ['gastric', 'esophageal']:
        genomic_notes.append(
            '🧬 HER2 positive — trastuzumab + chemotherapy (ToGA trial, OS benefit). '
            'Stage IV: trastuzumab + CAPOX or 5-FU/cisplatin (first-line standard). '
            'Second-line: trastuzumab deruxtecan (T-DXd, DESTINY-Gastric01).'
        )
        extra_regimens.append('Trastuzumab + CAPOX (HER2+ first-line — ToGA)')
        extra_regimens.append('Trastuzumab deruxtecan (T-DXd, 2nd line — DESTINY-Gastric01)')

    # KRAS mutant — colorectal (anti-EGFR contraindicated)
    if kras and cancer_type == 'colorectal':
        genomic_notes.append(
            '🧬 KRAS mutant — anti-EGFR therapy CONTRAINDICATED (cetuximab/panitumumab ineffective). '
            'Use bevacizumab-based regimens. KRAS G12C: sotorasib eligible (CodeBreak trial).'
        )
        # Remove any anti-EGFR mentions
        modified['regimens'] = [r for r in modified.get('regimens', [])
                                 if 'cetuximab' not in r.lower() and 'panitumumab' not in r.lower()]
        extra_regimens.append('Sotorasib (KRAS G12C specific — CodeBreak 300)')

    # BRAF V600E — colorectal
    if braf and cancer_type == 'colorectal':
        genomic_notes.append(
            '🧬 BRAF V600E detected — BEACON regimen: encorafenib + cetuximab (2nd line+). '
            'Poor prognosis marker. Intensive first-line (FOLFOXIRI + bevacizumab) if fit.'
        )
        extra_regimens.append('Encorafenib + cetuximab (BEACON-CRC, 2nd line)')
        extra_regimens.append('FOLFOXIRI + bevacizumab (1st line, fit BRAF V600E)')

    # BRCA1/2 — pancreatic
    if brca and cancer_type == 'pancreatic':
        genomic_notes.append(
            '🧬 BRCA1/2 germline mutation — PARP inhibitor eligible. '
            'Olaparib maintenance after ≥16 weeks platinum-based chemotherapy (POLO trial). '
            'Also sensitive to platinum-based regimens (FOLFIRINOX preferred).'
        )
        extra_regimens.append('Olaparib maintenance (BRCA1/2+ after platinum — POLO trial)')
        extra_regimens.append('FOLFIRINOX (platinum preferred in BRCA+)')

    # PD-L1 CPS ≥ 5 — gastric/esophageal
    if pdl1 >= 5 and cancer_type in ['gastric', 'esophageal']:
        genomic_notes.append(
            f'🧬 PD-L1 CPS = {pdl1} (≥5) — nivolumab + chemotherapy indicated (CheckMate-649). '
            'OS benefit demonstrated for CPS≥5. Consider pembrolizumab + chemo (KEYNOTE-590).'
        )
        extra_regimens.append(f'Nivolumab + FOLFOX/CAPOX (CPS={pdl1} ≥5 — CheckMate-649)')

    if extra_regimens:
        modified['regimens'] = modified.get('regimens', []) + extra_regimens

    modified['genomic_notes'] = genomic_notes if genomic_notes else ['No actionable genomic markers identified.']
    return modified

print('✅ Genomic marker integration loaded')
print('   Markers supported: MSI-H · HER2 · KRAS · BRAF V600E · BRCA1/2 · PD-L1 CPS')

✅ Genomic marker integration loaded
   Markers supported: MSI-H · HER2 · KRAS · BRAF V600E · BRCA1/2 · PD-L1 CPS


In [5]:
# Cell 5 — ClinicalTrials.gov API (free public API, no key needed)

def search_clinical_trials(cancer_type, stage, country='', max_results=5):
    print(f'\n🔍 Searching ClinicalTrials.gov for {cancer_type} Stage {stage}...')

    # Map cancer type to search term
    search_terms = {
        'colorectal': 'colorectal cancer',
        'gastric': 'gastric cancer stomach',
        'pancreatic': 'pancreatic cancer',
        'esophageal': 'esophageal cancer'
    }
    term = search_terms.get(cancer_type, cancer_type)
    stage_term = f'stage {stage}'

    # ClinicalTrials.gov API v2
    url = 'https://clinicaltrials.gov/api/v2/studies'
    params = {
        'query.cond': f'{term} {stage_term}',
        'filter.overallStatus': 'RECRUITING',
        'pageSize': max_results,
        'fields': 'NCTId,BriefTitle,Phase,LeadSponsorName,LocationCountry,EligibilityCriteria'
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        if response.status_code != 200:
            print(f'   ⚠️ API returned {response.status_code} — using offline fallback')
            return get_offline_trials(cancer_type, stage)

        data = response.json()
        studies = data.get('studies', [])

        if not studies:
            print('   ℹ️ No recruiting trials found — showing reference trials')
            return get_offline_trials(cancer_type, stage)

        trials = []
        for s in studies:
            proto = s.get('protocolSection', {})
            id_mod = proto.get('identificationModule', {})
            status_mod = proto.get('statusModule', {})
            design_mod = proto.get('designModule', {})
            sponsor_mod = proto.get('sponsorCollaboratorsModule', {})

            trials.append({
                'nct_id': id_mod.get('nctId', 'N/A'),
                'title': id_mod.get('briefTitle', 'N/A')[:120],
                'phase': design_mod.get('phases', ['N/A'])[0] if design_mod.get('phases') else 'N/A',
                'sponsor': sponsor_mod.get('leadSponsor', {}).get('name', 'N/A'),
                'status': status_mod.get('overallStatus', 'N/A'),
                'url': f'https://clinicaltrials.gov/study/{id_mod.get("nctId", "")}'
            })

        print(f'   ✅ Found {len(trials)} recruiting trials')
        return trials

    except Exception as e:
        print(f'   ⚠️ API error: {e} — using offline fallback')
        return get_offline_trials(cancer_type, stage)


def get_offline_trials(cancer_type, stage):
    # Landmark trials as fallback reference
    offline = {
        'colorectal': [
            {'nct_id': 'NCT04246814', 'title': 'KEYNOTE-177: Pembrolizumab vs chemo in MSI-H mCRC (1st line)', 'phase': 'PHASE3', 'sponsor': 'Merck', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT04246814'},
            {'nct_id': 'NCT01640327', 'title': 'BEACON CRC: Encorafenib + cetuximab in BRAF V600E mCRC', 'phase': 'PHASE3', 'sponsor': 'Array BioPharma', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT01640327'},
        ],
        'gastric': [
            {'nct_id': 'NCT02872116', 'title': 'CheckMate-649: Nivolumab + chemo in gastric/GEJ adenocarcinoma', 'phase': 'PHASE3', 'sponsor': 'BMS', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT02872116'},
            {'nct_id': 'NCT01668797', 'title': 'FLOT4: Perioperative FLOT vs ECF in gastric cancer', 'phase': 'PHASE3', 'sponsor': 'AIO', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT01668797'},
        ],
        'pancreatic': [
            {'nct_id': 'NCT02688348', 'title': 'POLO: Olaparib maintenance in BRCA+ metastatic pancreatic cancer', 'phase': 'PHASE3', 'sponsor': 'AstraZeneca', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT02688348'},
            {'nct_id': 'NCT01954784', 'title': 'MPACT: Gemcitabine + nab-paclitaxel in metastatic pancreatic cancer', 'phase': 'PHASE3', 'sponsor': 'Celgene', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT01954784'},
        ],
        'esophageal': [
            {'nct_id': 'NCT02811653', 'title': 'CheckMate-577: Nivolumab adjuvant after CRT + surgery in esophageal cancer', 'phase': 'PHASE3', 'sponsor': 'BMS', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT02811653'},
            {'nct_id': 'NCT04210115', 'title': 'KEYNOTE-590: Pembrolizumab + chemo in esophageal cancer', 'phase': 'PHASE3', 'sponsor': 'Merck', 'status': 'COMPLETED', 'url': 'https://clinicaltrials.gov/study/NCT04210115'},
        ]
    }
    return offline.get(cancer_type, [])

print('✅ ClinicalTrials.gov API ready (free public API — no key needed)')

✅ ClinicalTrials.gov API ready (free public API — no key needed)


In [6]:
# Cell 6 — Main Treatment Matcher

def run_treatment_matcher(
    cancer_type,
    stage,
    ecog,
    markers,
    patient_id='PATIENT_001',
    search_trials=True
):
    print('\n' + '=' * 60)
    print('  OncosenseAI — Module 3: Treatment Matcher')
    print('=' * 60)
    print(f'  Patient ID   : {patient_id}')
    print(f'  Cancer Type  : {cancer_type.upper()}')
    print(f'  Stage        : {stage}')
    print(f'  ECOG Score   : {ecog} — {ECOG_DESCRIPTIONS.get(ecog, "Unknown")}')
    print(f'  Markers      : {markers}')

    # 1. Get base protocol
    key = (cancer_type.lower(), stage.upper())
    if key not in TREATMENT_DB:
        print(f'\n❌ No protocol found for {cancer_type} Stage {stage}')
        return None

    protocol = dict(TREATMENT_DB[key])

    # 2. Apply ECOG de-escalation
    protocol = apply_ecog_deescalation(protocol, ecog)

    # 3. Apply genomic modifiers
    protocol = apply_genomic_modifiers(protocol, cancer_type.lower(), stage.upper(), markers)

    # 4. Search clinical trials
    trials = []
    if search_trials:
        trials = search_clinical_trials(cancer_type, stage)

    # 5. Display results
    print('\n' + '-' * 60)
    print('  TREATMENT RECOMMENDATION')
    print('-' * 60)
    print(f'\n  Primary    : {protocol["primary"]}')
    print(f'\n  Systemic   : {protocol["adjuvant"]}')
    print(f'\n  Regimens   :')
    for r in protocol.get('regimens', []):
        print(f'    • {r}')
    print(f'\n  Guideline  : {protocol["guideline"]}')
    print(f'\n  Notes      : {protocol["notes"]}')

    print('\n' + '-' * 60)
    print('  ECOG FITNESS ADJUSTMENT')
    print('-' * 60)
    print(f'  {protocol.get("ecog_note", "")}')

    print('\n' + '-' * 60)
    print('  GENOMIC MARKER FINDINGS')
    print('-' * 60)
    for note in protocol.get('genomic_notes', []):
        print(f'  {note}')

    if trials:
        print('\n' + '-' * 60)
        print('  CLINICAL TRIAL ELIGIBILITY (ClinicalTrials.gov)')
        print('-' * 60)
        for t in trials:
            print(f'\n  {t["nct_id"]} [{t["phase"]}] — {t["status"]}')
            print(f'  {t["title"]}')
            print(f'  🔗 {t["url"]}')

    print('\n' + '=' * 60)
    print('  ⚠️ DISCLAIMER: Research prototype only.')
    print('     Not for clinical use. Always apply clinical judgement.')
    print('=' * 60)

    # Build output
    output = {
        'patient_id': patient_id,
        'cancer_type': cancer_type,
        'stage': stage,
        'ecog': ecog,
        'markers': markers,
        'protocol': protocol,
        'clinical_trials': trials
    }
    return output

print('✅ Treatment matcher ready')

✅ Treatment matcher ready


In [7]:
# Cell 7 — Run the Treatment Matcher
# ✏️ Edit the parameters below to match your patient

# ── PATIENT INPUT ──────────────────────────────────────────────────────────
PATIENT_ID   = 'PATIENT_001'

CANCER_TYPE  = 'gastric'      # Options: colorectal | gastric | pancreatic | esophageal

STAGE        = 'III'          # Options: I | II | III | IV

ECOG_SCORE   = 1              # 0=fully active, 1=light work, 2=self-care, 3=limited, 4=bedbound

MARKERS = {
    'MSI_H'   : False,   # MSI-H / dMMR status
    'HER2'    : True,    # HER2 positive (gastric/esophageal)
    'KRAS'    : False,   # KRAS mutant (colorectal)
    'BRAF'    : False,   # BRAF V600E (colorectal)
    'BRCA'    : False,   # BRCA1/2 germline (pancreatic)
    'PDL1_CPS': 8        # PD-L1 CPS score (gastric/esophageal)
}
# ───────────────────────────────────────────────────────────────────────────

result = run_treatment_matcher(
    cancer_type=CANCER_TYPE,
    stage=STAGE,
    ecog=ECOG_SCORE,
    markers=MARKERS,
    patient_id=PATIENT_ID,
    search_trials=True
)


  OncosenseAI — Module 3: Treatment Matcher
  Patient ID   : PATIENT_001
  Cancer Type  : GASTRIC
  Stage        : III
  ECOG Score   : 1 — Restricted in strenuous activity — ambulatory, light work
  Markers      : {'MSI_H': False, 'HER2': True, 'KRAS': False, 'BRAF': False, 'BRCA': False, 'PDL1_CPS': 8}

🔍 Searching ClinicalTrials.gov for gastric Stage III...
   ✅ Found 5 recruiting trials

------------------------------------------------------------
  TREATMENT RECOMMENDATION
------------------------------------------------------------

  Primary    : Perioperative FLOT4 + D2 gastrectomy

  Systemic   : FLOT4 x4 cycles pre-op + surgery + FLOT4 x4 cycles post-op

  Regimens   :
    • FLOT4 (standard)
    • CAPOX (alternative)
    • Nivolumab + chemotherapy (HER2-negative, CPS≥5)
    • Trastuzumab + CAPOX (HER2+ first-line — ToGA)
    • Trastuzumab deruxtecan (T-DXd, 2nd line — DESTINY-Gastric01)
    • Nivolumab + FOLFOX/CAPOX (CPS=8 ≥5 — CheckMate-649)

  Guideline  : NCCN Gastric v2

In [8]:
# Cell 8 — Save module3_output.json

if result:
    output_path = '/content/module3_output.json'
    with open(output_path, 'w') as f:
        json.dump(result, f, indent=2, default=str)
    print(f'✅ Saved: {output_path}')
    files.download(output_path)
    print('📥 module3_output.json downloaded')
else:
    print('❌ No result to save — check cancer type and stage inputs in Cell 7')

✅ Saved: /content/module3_output.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 module3_output.json downloaded


---
## Cell 9 — Full Pipeline Integration
Load Module 1 + Module 2 outputs and combine with Module 3 into a single patient report.

In [9]:
# Cell 9 — Full Pipeline: Module 1 + 2 + 3 combined
# Upload module2_output.json when prompted (optional)

print('Upload module2_output.json (or Cancel to skip):')
try:
    uploaded = files.upload()
    if uploaded:
        m2 = json.loads(list(uploaded.values())[0])
        m2i = m2.get('module1_integration', {})
        visual_score = m2i.get('visual_concern_score', 0.0)
        highest_concern = m2i.get('highest_concern', 'Low')
        flag_urgent = m2i.get('flag_urgent', False)
        print(f'\n✅ Module 2 loaded')
        print(f'   Visual concern score : {visual_score}')
        print(f'   Highest concern      : {highest_concern}')
        print(f'   Urgent flag          : {flag_urgent}')
    else:
        print('   ⏭️  Skipping Module 2 integration')
        visual_score, highest_concern, flag_urgent = 0.0, 'Low', False
except Exception as e:
    print(f'   ⚠️ Could not load Module 2: {e}')
    visual_score, highest_concern, flag_urgent = 0.0, 'Low', False

# Combined summary
print('\n' + '=' * 60)
print('  OncosenseAI — Full Pipeline Summary')
print('=' * 60)
print(f'  Patient        : {PATIENT_ID}')
print(f'  Cancer         : {CANCER_TYPE.upper()} Stage {STAGE}')
print(f'  ECOG           : {ECOG_SCORE}')
print(f'  Visual Score   : {visual_score} ({highest_concern} concern)')
print(f'  Urgent Flag    : {flag_urgent}')
if result:
    print(f'\n  Treatment      : {result["protocol"]["primary"]}')
    print(f'  Regimen        : {result["protocol"]["adjuvant"]}')
    print(f'  Guideline      : {result["protocol"]["guideline"]}')
print('\n  ⚠️ Research prototype only — not for clinical use')
print('=' * 60)

Upload module2_output.json (or Cancel to skip):


Saving module2_output.json to module2_output.json

✅ Module 2 loaded
   Visual concern score : 0.233
   Highest concern      : Moderate
   Urgent flag          : False

  OncosenseAI — Full Pipeline Summary
  Patient        : PATIENT_001
  Cancer         : GASTRIC Stage III
  ECOG           : 1
  Visual Score   : 0.233 (Moderate concern)
  Urgent Flag    : False

  Treatment      : Perioperative FLOT4 + D2 gastrectomy
  Regimen        : FLOT4 x4 cycles pre-op + surgery + FLOT4 x4 cycles post-op
  Guideline      : NCCN Gastric v2.2024 / ESMO 2022 / CheckMate-649

  ⚠️ Research prototype only — not for clinical use


---
## Status

| Component | Status |
|-----------|--------|
| Treatment protocol DB (NCCN/NICE/ESMO) | ✅ Built |
| ECOG performance status de-escalation | ✅ Built |
| Genomic marker integration (MSI-H, HER2, KRAS, BRAF, BRCA, PD-L1) | ✅ Built |
| ClinicalTrials.gov API | ✅ Built |
| Module 1+2+3 pipeline integration | ✅ Built |
| PDF clinical report (Module 4) | 📋 Next |

---
*OncosenseAI · Module 3 · Treatment Matcher*  
*Built by a physician. For clinicians. For patients.*  
> ⚠️ Research Prototype Only. Not validated for clinical use.